In [ ]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import joblib

In [ ]:
df = pd.read_csv("fire_incidents.csv", encoding='latin1')

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['Date'])

In [ ]:
daily_incidents = df.groupby(df['Date'].dt.date).size()
daily_incidents.index = pd.to_datetime(daily_incidents.index)
daily_incidents = daily_incidents.sort_index()

print(f"Data range: {daily_incidents.index[0].date()} → {daily_incidents.index[-1].date()}")

Data range: 2018-01-14 → 2024-05-17


In [ ]:
# Keep only 2022
daily_incidents = daily_incidents[daily_incidents.index.year == 2022]

In [ ]:
# 80/20 split
total_size = len(daily_incidents)
train_size = int(total_size * 0.8)
train = daily_incidents[:train_size]
test = daily_incidents[train_size:]

print(f"\n80/20 Split → Train: {len(train)} days | Test: {len(test)} days")


80/20 Split → Train: 108 days | Test: 28 days


In [ ]:
print("\nTraining ARIMA(2,1,2)...")
model = ARIMA(train, order=(2,1,2))
model_fit = model.fit()


Training ARIMA(2,1,2)...


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [ ]:
# Forecast test period
forecast = model_fit.forecast(steps=len(test))

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [ ]:
# Metrics
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
mape = np.mean(np.abs((test - forecast) / (test + 1))) * 100

print(f"\n80/20 Results:")
print(f"MAE : {mae:.2f} incidents/day")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.1f}%")


80/20 Results:
MAE : 0.58 incidents/day
RMSE: 0.74
MAPE: nan%


/tmp/ipython-input-3996569147.py:4: RuntimeWarning: '<' not supported between instances of 'int' and 'Timestamp', sort order is undefined for incomparable objects.
  mape = np.mean(np.abs((test - forecast) / (test + 1))) * 100


In [ ]:
# Retrain on full data for final model
final_model = ARIMA(daily_incidents, order=(2,1,2))
final_fit = final_model.fit()

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [ ]:
joblib.dump(final_fit, 'fire_arima_80_20.pkl')
print("\nModel saved as fire_arima_80_20.pkl")


Model saved as fire_arima_80_20.pkl
